# DuocAIAgent · Orientador Duoc (v2)

Notebook para correr el agente **paso a paso**. Usa el mismo código de `src/` y `prompts/` (lo que corre la app de Streamlit),
así lo que se prueba acá es exactamente lo que se entrega.

**Pipeline:** PDFs + web → limpieza → chunking → embeddings → PostgreSQL/pgvector → búsqueda + reranking → prompt → Llama 3.1 8B (Groq) → respuesta

Antes de correrlo:
1. Tener el `.env` con `GROQ_API_KEY` y `DATABASE_URL` (copiar de `.env.example`).
2. Tener PostgreSQL con pgvector arriba (`docker compose up -d db`).
3. Abrir este notebook desde la **carpeta raíz del repo** (donde está `app.py`).

Ver `docs/INSTRUCTIVO.md` para el paso a paso completo.

## 0. Configuración

In [ ]:
import os, sys
# para poder importar src/ y prompts/ aunque Jupyter se haya abierto desde otra carpeta
if not os.path.exists("app.py") and os.path.exists("../app.py"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from src import config

print("Carpeta de trabajo:", os.getcwd())
print("Modelo LLM:        ", config.GROQ_MODEL)
print("Groq API key:      ", "OK" if config.GROQ_API_KEY and config.GROQ_API_KEY.startswith("gsk_") else "FALTA (revisar .env)")
print("Base de datos:     ", config.DATABASE_URL.split("@")[-1])   # no mostramos la clave
print("Prompt por defecto:", config.PROMPT_VERSION)

In [ ]:
# Probamos la conexion a PostgreSQL y creamos las tablas (documentos + interacciones)
from src.utils.postgres import PostgresClient

db = PostgresClient()
db.crear_tablas()
print("Conexion OK. Version:", db.conn.execute("SELECT version()").fetchone()[0][:40])
print("Extension vector:", db.conn.execute("SELECT extversion FROM pg_extension WHERE extname = 'vector'").fetchone())

## 1. Extracción y limpieza del texto (fuentes internas)

En la v1 pypdf devolvía el calendario palabra por palabra (`FUENTE\n \nDE\n \n...`). Acá se ve el antes y el después de `limpiar_texto()`.

In [ ]:
from pathlib import Path
from pypdf import PdfReader
from src.ingesta.ingest import limpiar_texto, extraer_texto_pdf

ruta_cal = Path(config.CARPETA_INTERNOS) / "calendario DuocUC.pdf"
crudo = PdfReader(ruta_cal).pages[0].extract_text()
print("=== ANTES (pypdf) ===")
print(repr(crudo[:200]))
print("\n=== DESPUES (limpiar_texto) ===")
print(limpiar_texto(crudo)[:400])

## 2. Chunking por tipo de documento

Cada documento usa su propio tamaño de chunk (ver `CONFIG_FUENTES` en `src/ingesta/ingest.py` y `docs/decisiones.md`).

In [ ]:
from src.ingesta.ingest import config_para_archivo, dividir_en_chunks

chunks_por_doc = {}
for pdf in sorted(Path(config.CARPETA_INTERNOS).glob("*.pdf")):
    cfg = config_para_archivo(pdf.name)
    chunks = dividir_en_chunks(extraer_texto_pdf(pdf), cfg)
    chunks_por_doc[cfg["fuente"]] = chunks
    largos = [len(c) for c in chunks]
    print(f"   -> {cfg['fuente']}: {len(chunks)} chunks (size={cfg['chunk_size']}, overlap={cfg['chunk_overlap']}, min={min(largos)}, max={max(largos)})\n")

In [ ]:
# Ejemplo: en el calendario cada chunk lleva su seccion (TAV, 2026-1, 2026-2)
for c in chunks_por_doc["calendario_academico"][1:4]:
    print(c[:300], "\n" + "-" * 80)

In [ ]:
# Ejemplo: el reglamento queda cortado por articulo
for c in chunks_por_doc["reglamento_academico"][20:23]:
    print(c[:300], "\n" + "-" * 80)

## 3. Embeddings + carga a PostgreSQL (ingesta)

`Ingestor.ingestar_todo()` hace los pasos 1 y 2 para los PDFs y para las URLs de `data/externos/fuentes.txt`,
calcula los embeddings (`paraphrase-multilingual-MiniLM-L12-v2`, 384 dim) y los guarda en la tabla `documentos`.

La primera vez descarga el modelo de embeddings (~470 MB), así que se demora un poco. Si ya estaba cargado no duplica nada.
Para recargar todo usar `reset=True`.

In [ ]:
from src.ingesta.ingest import Ingestor

ingestor = Ingestor(db=db)
ingestor.ingestar_todo(reset=False)

In [ ]:
# Que quedo en la BD
for fila in db.conn.execute("SELECT tipo, fuente, COUNT(*) FROM documentos GROUP BY tipo, fuente ORDER BY tipo, fuente"):
    print(fila)

## 4. Retrieval: búsqueda vectorial + reranking

Comparamos lo que trae pgvector solo (similitud coseno) contra lo que queda después del cross-encoder.

In [ ]:
from src.retrieval.retrieval import Retriever

pregunta = "¿Cuándo es la inscripción de asignaturas del segundo semestre?"

sin_rerank = Retriever(db=db, usar_reranker=False).retrieve(pregunta, top_k=5)
con_rerank = Retriever(db=db, usar_reranker=True).retrieve(pregunta, top_k=5)

print("=== Solo pgvector (coseno) ===")
for c in sin_rerank:
    print(f"{c['score']:.3f} | {c['fuente']:22s} | {c['contenido'][:90]!r}")
print("\n=== pgvector top-20 + reranker -> top-5 ===")
for c in con_rerank:
    print(f"{c['rerank_score']:.3f} | {c['fuente']:22s} | {c['contenido'][:90]!r}")

## 5. Prompts

El system prompt sigue la estructura de la clase 1.2: **rol → tarea → contexto → formato → restricciones**.
Hay 3 versiones: `zero_shot`, `few_shot` (por defecto) y `cot`.

In [ ]:
from prompts.prompt import construir_system_prompt, PROMPTS
from src.generate.generate import formatear_contexto, fecha_hoy

print(construir_system_prompt("few_shot", formatear_contexto(con_rerank[:2]), fecha_hoy()))

## 6. Generación con Llama 3.1 8B (Groq)

### 6.1 Zero-shot vs few-shot vs chain-of-thought

Misma pregunta y mismo contexto, cambia solo el prompt.

In [ ]:
from src.generate.generate import RAGGenerator
from src.utils.llm import GroqLLM

retriever = Retriever(db=db)
llm = GroqLLM()
agentes = {v: RAGGenerator(prompt_version=v, retriever=retriever, llm=llm) for v in PROMPTS}

preguntas = [
    "¿Cuántas horas dura la práctica laboral?",
    "¿Cuántas horas dura la práctica profesional?",   # no esta en los documentos -> no deberia inventar
    "¿Hasta cuándo puedo retirar asignaturas en el segundo semestre?",
    "Dime los cinco mejores juegos de God of War",    # fuera de dominio
]

for p in preguntas:
    chunks = retriever.retrieve(p)          # un solo retrieve, se comparte entre las 3 versiones
    print("=" * 100, f"\nPREGUNTA: {p}\n")
    for version, agente in agentes.items():
        r = agente.generate(p, chunks=chunks, session_id="notebook")
        print(f"[{version}] ({r['total_tokens']} tokens, {r['latencia_ms']} ms)\n{r['answer']}\n")

In [ ]:
# En la version CoT el estudiante solo ve la "Respuesta final", pero podemos ver el razonamiento completo:
r = agentes["cot"].generate("¿Cuánto falta para que empiecen las clases del segundo semestre?", session_id="notebook")
print(r["raw_answer"])

### 6.2 Conversación con historial

In [ ]:
agente = agentes["few_shot"]
historial = []

for p in ["Hola! entré este año a Duoc", "¿Qué porcentaje de asistencia necesito?", "¿Y qué pasa si no lo cumplo?"]:
    r = agente.generate(p, history=historial, session_id="notebook-conversacion")
    print(f"Estudiante: {p}\nAgente: {r['answer']}\n")
    historial += [{"role": "user", "content": p}, {"role": "assistant", "content": r["answer"]}]

In [ ]:
# Fuentes que uso la ultima respuesta (esto es lo que se muestra en la app en "Fuentes recuperadas")
for s in r["sources"]:
    print(f"{s['score']:.3f} | {s['fuente']} ({s['tipo']}) | {s['origen']}")

## 7. Trazabilidad

Cada respuesta queda guardada en la tabla `interacciones` (pregunta, respuesta, fuentes, prompt, tokens y latencia).

In [ ]:
filas = db.conn.execute("""
    SELECT creado_en, prompt_version, prompt_tokens + completion_tokens AS tokens, latencia_ms, LEFT(pregunta, 50)
    FROM interacciones ORDER BY creado_en DESC LIMIT 10
""").fetchall()
for f in filas:
    print(f)

## 8. Evaluación

Corre las 14 preguntas de `eval/dataset.json` y un juez (Llama 3.3 70B) evalúa **correctitud** y **fundamentación**.
También mide si el retrieval trajo la fuente correcta (**hit_fuente**).

Ojo: con las 3 versiones son ~84 llamadas a Groq (unos 3 a 5 minutos por el límite del tier gratis).
Para probar rápido se puede dejar solo `["few_shot"]`.
Los resultados quedan en `eval/resultados/` (JSON + tabla .md) para el informe.

In [ ]:
from eval.evaluate import run_evaluation

salida = run_evaluation(["zero_shot", "few_shot", "cot"])

print("\n=== Resumen ===")
print(f"{'prompt':10s} {'correctitud':>12s} {'fundament.':>11s} {'hit_fuente':>11s} {'tokens':>8s}")
for v, m in salida["resumen"].items():
    print(f"{v:10s} {m['correctitud']!s:>12s} {m['fundamentacion']!s:>11s} {m['hit_fuente']!s:>11s} {m['tokens_promedio']!s:>8s}")

In [ ]:
# Preguntas donde le fue peor (para analizar en el informe)
malas = sorted([x for x in salida["detalle"] if x["correctitud"] is not None], key=lambda x: x["correctitud"])[:5]
for x in malas:
    print(f"[{x['prompt_version']}] {x['pregunta']}\n  esperada: {x['respuesta_esperada']}\n  agente:   {x['respuesta'][:200]}\n  juez:     {x['comentario_juez']}\n")

## 9. Tests automáticos

In [ ]:
!{sys.executable} -m pytest -q